In [1]:
import os
import sys

os.environ["SPARK_MAJOR_VERSION"] = "3"
os.environ["SPARK_HOME"] = "/usr/sdp/current/spark3-client/"
os.environ["PYSPARK_PYTHON"] = "/opt/sdp/mlpy3811v23/bin/python"
os.environ["PYSPARK_DRIVER_PYTHON"] = "/opt/sdp/mlpy3811v23/bin/python"
os.environ["LD_LIBRARY_PATH"] = "/opt/python/virtualenv/jupyter/lib"
sys.path.insert(0, "/usr/sdp/current/spark3-client/python/")
sys.path.insert(0, "/usr/sdp/current/spark3-client/python/lib/py4j-0.10.9.3-src.zip")

import yaml
from functools import reduce
from itertools import chain
import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm
from dateutil.relativedelta import relativedelta
import datetime
from collections import defaultdict
import pyspark
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.feature import MinMaxScaler

from pyspark.sql import functions as F, types as T, DataFrame
from pyspark.sql.window import Window
from pyspark.sql.types import MapType, StringType, IntegerType, DoubleType, ByteType
from pyspark.sql.functions import from_json
from pyspark.sql import SparkSession
from pyspark import SparkConf

from dataclasses import dataclass
from IPython.display import display, clear_output
from typing import List, Union, Callable
import subprocess
import time

from pathlib import Path
from git import Repo



session = (
    SparkSession.builder
        .appName("uplift_modeling_s_learner")
        .master("yarn")
        .config("spark.sql.shuffle.partitions", "100") 
        .config("spark.executor.instances", "15") ## !!!!!
        .config("spark.driver.memory", "10g")
        .config("spark.executor.memory",  "20g")                        
        .config("spark.executor.cores", "4")
        .config("spark.driver.maxResultSize", "10g")
        .config("spark.hadoop.hive.exec.dynamic.partition", "true")
        .config("spark.hadoop.hive.exec.dynamic.partition.mode", "nonstrict")
        .config("spark.shuffle.service.enabled", "true")
        .config("spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive", "true")
        .config("spark.sql.sources.partitionOverwriteMode", "dynamic")
        .config("spark.sql.broadcastTimeout", "300")
        .config("spark.port.maxRetries", "150")
        .config("spark.shuffle.memoryFraction", "0.5")
        .config("spark.sql.legacy.timeParserPolicy","LEGACY")
        .config("spark.kryoserializer.buffer.max", "1536")
        .config("spark.dynamicAllocation.enabled", "false")
        .config("spark.sql.parquet.int96RebaseModeInRead", "CORRECTED")
        .config("spark.sql.parquet.int96RebaseModeInWrite", "CORRECTED")
        .config("spark.hadoop.parquet.block.size", "134217728")
        .config("spark.sql.autoBroadcastJoinThreshold", "-1")
        .config("spark.scheduler.allocation.file", "hdfs://arnsdpsbx/user/team/team_ai_avatar/hive/utils/fairScheduleConfig.xml")
        .config("spark.scheduler.mode", "FAIR")
        .enableHiveSupport()
)

spark = session.getOrCreate()  
spark.sparkContext.setLogLevel("ERROR")

clear_output()

In [4]:
#!pwd - /data/hdfs/data/workspace/21937299_omega-sbrf-ru/notebooks/avatar_fm/examples/td_response

sys.path.insert(0, "../../../") # path to avatar directory
from avatar.preprocessing.spark.pipeline import TabularPreprocessor


def ft_aggr_mnth_to_target(features_df, tgt_df):
    if "bucket_ratio" in features_df.columns:
        features_df = features_df.drop("bucket_ratio")
    
    target_df = tgt_df.withColumn(
        "month_part", F.last_day(F.add_months("report_month", -2))
    )
    
    features_tgt = (
        features_df
        .join(
            target_df,
            on=["epk_id", "report_month", "month_part", "bucket_num"],
            how="inner"
        )
    )

    return features_tgt

## Описание источников
1. `hdfs://arnsdpsbx/user/team/team_ai_avatar/hive/datasets/td_response/targets` - таргеты по продукту `Вклады`
* **epk_id**
* **target_attr_1** - факт
* **target_attr_2** - Канал коммуникации; `{0: SMS, 1: Push, 2: ERKC, 3: Robot, 4: VKC}`
* **target_attr_3** - Флаг контрольной группы (0 - ЦГ, 1 - КГ)
* **target_attr_4** - флаг калибровочной кампании
* **report_month** - месяц кампании
  
---

2. `hdfs://arnsdpsbx/user/team/team_ai_avatar/hive/datasets/cc_response/ft_aggr_mnth_clean` - часть мегавитрины (~250 признаков)
* **month_part** - месяц агрегатов

---

3. `hdfs://arnsdpsbx/user/team/team_ai_avatar/ds/rusakov/may_pilot/cc_sa_td_seq/20_epoch_seq_states_fp32` - FM hidden_states - for clients

In [6]:
# Reading categorical and numeric aggr_mnth columns

with open("artifacts/ft_aggr_mnth_columns.yaml", "r") as f:
    ft_aggr_mnth_columns = yaml.safe_load(f)

# Dates for train, valid, and test
train_targets_upper_date = F.last_day(F.lit('2024-09-01')).cast(T.DateType()) #datetime.date(2024, 9, 30)
valid_targets_date = F.last_day(F.lit('2024-10-01')).cast(T.DateType()) # datetime.date(2024, 10, 31)
test_targets_date = F.last_day(F.lit('2024-12-01')).cast(T.DateType()) # datetitme.date(2024, 12, 31)


# Join витрины агрегатов left - target, right - aggr_mnths
targets = (
    spark.read.parquet(
        "hdfs://arnsdpsbx/user/team/team_ai_avatar/hive/datasets/td_response/targets"
    )
    .where(F.col("target_attr_2") != 1)
)

targets.show(1)

ft_aggr_mnth = (
    spark.read.parquet(
        "hdfs://arnsdpsbx/user/team/team_ai_avatar/hive/datasets/td_response/ft_aggr_mnth_clean"
    )
)

targets_ft_aggr_mnth = ft_aggr_mnth_to_target(
    features_df=ft_aggr_mnth,
    tgt_df=targets
)

+-------------------+-------------+-------------+-------------+-------------+------------+----------+------------+
|             epk_id|target_attr_1|target_attr_2|target_attr_3|target_attr_4|bucket_ratio|bucket_num|report_month|
+-------------------+-------------+-------------+-------------+-------------+------------+----------+------------+
|1128613305888242923|            0|            0|            0|            0|  0.01074887|        11|  2025-02-28|
+-------------------+-------------+-------------+-------------+-------------+------------+----------+------------+
only showing top 1 row



In [7]:
# Fit and transform data using TabularPreprocessor

tabular_preprocessor = TabularPreprocessor(
    categorical_columns=ft_aggr_mnth_columns["categorical_columns"], # ft_aggr_mnth cat_cols + target channel_type and is_control flag 
    numeric_columns=ft_aggr_mnth_columns["numeric_columns"],
)

processed_df = (
    tabular_preprocessor
    .fit_transform(
        df=targets_ft_aggr_mnth
    )
)

processed_df.printSchema()

processed_df = processed_df.select(
    "epk_id",
    "report_month",
    "month_part",
    "bucket_num",
    "target_attr_1",
    "target_attr_2",
    "target_attr_3",
    "cat_features",
    "num_features"
)

processed_df.printSchema()

root
 |-- epk_id: long (nullable = true)
 |-- report_month: date (nullable = true)
 |-- month_part: date (nullable = true)
 |-- bucket_num: integer (nullable = true)
 |-- evt_dttm: date (nullable = true)
 |-- target_attr_1: integer (nullable = true)
 |-- target_attr_2: integer (nullable = true)
 |-- target_attr_3: integer (nullable = true)
 |-- target_attr_4: integer (nullable = true)
 |-- bucket_ratio: decimal(10,8) (nullable = true)
 |-- cat_features: array (nullable = false)
 |    |-- element: long (containsNull = true)
 |-- num_features: array (nullable = false)
 |    |-- element: float (containsNull = true)

root
 |-- epk_id: long (nullable = true)
 |-- report_month: date (nullable = true)
 |-- month_part: date (nullable = true)
 |-- bucket_num: integer (nullable = true)
 |-- target_attr_1: integer (nullable = true)
 |-- target_attr_2: integer (nullable = true)
 |-- target_attr_3: integer (nullable = true)
 |-- cat_features: array (nullable = false)
 |    |-- element: long (contai

In [8]:
# Join hidden_states to tab_features

hidden_states = (
    spark.read.parquet(
        "hdfs://arnsdpsbx/user/team/team_ai_avatar/avatar_fm/examples/demo_hidden_states",
    )
    .select("epk_id", "report_month", "seq_hidden_state")
    .withColumn("report_month", F.col("report_month").cast(T.DateType()))
    .withColumn("month_part", F.last_day(F.add_months(F.col("report_month"), -2)))
)

# collect hidden_size
hidden_size = (
    hidden_states
    .select("seq_hidden_state")
    .limit(1)
    .withColumn("hidden_size", F.size("seq_hidden_state"))
    .select("hidden_size")
    .collect()[0][0]
)

# Join hidden_states to processed_df
ft_aggr_seq_states = processed_df.join(
    hidden_states,
    how="left",
    on=["epk_id", "month_part", "report_month"]
)

# Fillna null values
final_tab_features = ft_aggr_seq_states.withColumn(
    "seq_hidden_state",
    F.when(
        F.col("seq_hidden_state").isNull(),
        F.array([F.lit(0.0) for _ in range(hidden_size)])  # Creates array of 128 zeros
    ).otherwise(F.col("seq_hidden_state"))
)

In [9]:
# train valid test split

# train
train = (
    final_tab_features
    .where(F.col("report_month") <= train_targets_upper_date)
)

# valid
valid = (
    final_tab_features
    .where(F.col("report_month") == valid_targets_date)
)

# test
test = (
    final_tab_features
    .where(F.col("report_month") == test_targets_date)
)

In [10]:
(
    train
    .write
    .mode("overwrite")
    .parquet(
        "hdfs://arnsdpsbx/user/team/team_ai_avatar/avatar_fm/examples/uplift_modeling/s_learner/train"
    )
)

(
    valid
    .write
    .mode("overwrite")
    .parquet(
        "hdfs://arnsdpsbx/user/team/team_ai_avatar/avatar_fm/examples/uplift_modeling/s_learner/valid"
    )
)

(
    test
    .write
    .mode("overwrite")
    .parquet(
        "hdfs://arnsdpsbx/user/team/team_ai_avatar/avatar_fm/examples/uplift_modeling/s_learner/test"
    )
)

In [11]:
# Number of unique values in categorical columns, number of numeric columns
# This values required for model configuration

tabular_preprocessor.vocab_size, len(tabular_preprocessor.num_cols)

(167, 189)

In [12]:
# Dump, load tabular preprocessor

# dump
preprocessor_config = tabular_preprocessor.dump()

with open("artifacts/td_tabular_preprocessor.yaml", "w") as f:
    yaml.dump(preprocessor_config, f)

# load
with open("artifacts/td_tabular_preprocessor.yaml", "r") as f:
    loaded_cfg = yaml.safe_load(f)

loaded_preprocessor = TabularPreprocessor.load(preprocessor_config)

In [13]:
spark.stop()